In [54]:
from pathlib import Path
import pandas as pd


## Traficom M1 preprocessing
This notebook preprocesses Traficom M1 passenger car registry data and builds cleaned brand-level and model-level summary tables for later merging with the spare-parts pricing dataset.

In [ ]:
path = "/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/traficom_datasets/TieliikenneAvoinData_31_12_2025.csv"
output_dir = Path("datasets/traficom_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(path, sep=";", encoding="latin1", low_memory=False)

## Load and inspect data
Read the Traficom extract and inspect the main fields before narrowing to M1 cars.

## Notebook workflow
1. Load and inspect Traficom data
2. Filter to M1 passenger cars
3. Clean and standardize key variables
4. Create cleaned grouping keys for brand and model family
5. Build `brand_summary`
6. Build `model_summary`
7. Inspect summary outputs for target models

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.info())

(5122260, 27)
['merkkiSelvakielinen', 'mallimerkinta', 'kaupallinenNimi', 'variantti', 'versio', 'ensirekisterointipvm', 'kayttoonottopvm', 'ajoneuvoluokka', 'ajoneuvoryhma', 'korityyppi', 'kayttovoima', 'yksittaisKayttovoima', 'sahkohybridi', 'sahkohybridinluokka', 'iskutilavuus', 'suurinNettoteho', 'sylintereidenLkm', 'omamassa', 'vaihteisto', 'vaihteidenLkm', 'ahdin', 'ovienLukumaara', 'istumapaikkojenLkm', 'voimanvalJaTehostamistapa', 'matkamittarilukema', 'brand', 'model']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5122260 entries, 0 to 5122259
Data columns (total 27 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   merkkiSelvakielinen        string        
 1   mallimerkinta              string        
 2   kaupallinenNimi            string        
 3   variantti                  string        
 4   versio                     string        
 5   ensirekisterointipvm       datetime64[ns]
 6   kayttoonottopvm        

In [ ]:
cols_keep = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
    "ensirekisterointipvm",
    "kayttoonottopvm",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteisto",
    "vaihteidenLkm",
    "ahdin",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "voimanvalJaTehostamistapa",
    "matkamittarilukema"
]

traficom_reduced = df[cols_keep].copy()

print(traficom_reduced.shape)
traficom_reduced.head()

# Keep the reduced table in memory for now; no exports yet.


(5122260, 25)


In [ ]:
missing = (
    traficom_reduced.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
)

print(missing)

sahkohybridinluokka          0.924085
vaihteidenLkm                0.588855
sahkohybridi                 0.570312
ajoneuvoryhma                0.542370
ovienLukumaara               0.520365
vaihteisto                   0.486266
ahdin                        0.474071
matkamittarilukema           0.448032
variantti                    0.374765
korityyppi                   0.359669
versio                       0.339221
sylintereidenLkm             0.331299
suurinNettoteho              0.330044
iskutilavuus                 0.265691
kaupallinenNimi              0.231678
istumapaikkojenLkm           0.231567
yksittaisKayttovoima         0.230334
kayttovoima                  0.230329
voimanvalJaTehostamistapa    0.195374
mallimerkinta                0.178238
ensirekisterointipvm         0.030804
omamassa                     0.002126
merkkiSelvakielinen          0.000360
kayttoonottopvm              0.000000
ajoneuvoluokka               0.000000
Name: missing_share, dtype: float64


In [ ]:
cat_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "vaihteisto",
    "ahdin",
]

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(traficom_reduced[col].value_counts(dropna=False).head(15))


--- merkkiSelvakielinen ---
merkkiSelvakielinen
Toyota            442877
Volvo             295934
Ford              264593
Mercedes-Benz     263777
Volkswagen, VW    189455
Muuli             180017
Skoda             174880
Volkswagen        172155
BMW               147263
Nissan            143570
Omavalmiste       129396
Audi              125486
Kia               118342
Opel              110207
Honda              92451
Name: count, dtype: int64

--- mallimerkinta ---
mallimerkinta
NaN                                                   912979
TOYOTA COROLLA Farmari (AC) 4ov 1798cm3                17589
XC60 Farmari (AC) 5ov 1969cm3 A                        16893
1250 XI/250                                            16596
TRANSPORTER Umpikorinen (BB) 6ov 1968cm3               16537
TOYOTA RAV4 Farmari (AC) 4ov 2487cm3                   15669
TOYOTA AURIS Monikäyttöajoneuvo (AF) 4ov 1798cm3       15447
Nissan Qashqai Monikäyttöajoneuvo (AF) 4ov 1197cm3     14527
TOYOTA YARIS Viistoperä (

ajoneuvoryhma
NaN            2778158
1, 13           603755
74              183494
75, 509         156317
2, 13           145061
109             138175
75               97355
509              94253
13, 20, 513      78761
6, 13            65029
107              62398
75, 933          49134
9, 13            47708
21               44689
7, 13            41542
Name: count, dtype: int64

--- korityyppi ---
korityyppi
NaN    1842320
AC     1064900
AB      529154
AF      512118
AA      413817
DC      358807
BB      223340
BA       34865
DA       21689
AD       20532
DB       14291
BD       13757
SA       12819
SE       12381
SG       10278
Name: count, dtype: int64

--- kayttovoima ---
kayttovoima
01     2261333
02     1474684
NaN    1179805
04      177816
13        9491
38        8592
40        4601
48        1667
31        1143
39         982
34         487
03         429
11         309
67         222
65         214
Name: count, dtype: int64

--- yksittaisKayttovoima ---
yksittaisKayttovoim

## Clean key variables
Parse dates, clean types, and create stable text variables for later summaries.

In [ ]:
df = traficom_reduced.copy()

# Traficom uses two different raw date formats here:
# ensirekisterointipvm is typically dd.mm.yyyy, while kayttoonottopvm is yyyymmdd.
# The previous generic to_datetime call parsed yyyymmdd integers as Unix ns timestamps,
# which collapsed use_year to 1970 and inflated vehicle_age.
date_cols = ["ensirekisterointipvm", "kayttoonottopvm"]
raw_date_sample = df[date_cols].head(10).copy()

df["ensirekisterointipvm"] = pd.to_datetime(
    df["ensirekisterointipvm"].astype("string").str.strip(),
    format="%d.%m.%Y",
    errors="coerce",
)

raw_use_date = df["kayttoonottopvm"].astype("string").str.strip()
raw_use_date = raw_use_date.str.replace(r"\\.0$", "", regex=True)
raw_use_date = raw_use_date.where(raw_use_date.str.len() == 8, raw_use_date.str.zfill(8))
raw_use_date = raw_use_date.where(~raw_use_date.str.match(r"^\\d{4}0000$"), raw_use_date.str[:4] + "0101")
df["kayttoonottopvm"] = pd.to_datetime(raw_use_date, format="%Y%m%d", errors="coerce")

num_cols = [
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteidenLkm",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "matkamittarilukema",
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df[["ensirekisterointipvm", "kayttoonottopvm"] + num_cols].info()
df[num_cols].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5122260 entries, 0 to 5122259
Data columns (total 10 columns):
 #   Column                Dtype         
---  ------                -----         
 0   ensirekisterointipvm  datetime64[ns]
 1   kayttoonottopvm       datetime64[ns]
 2   iskutilavuus          float64       
 3   suurinNettoteho       float64       
 4   sylintereidenLkm      float64       
 5   omamassa              float64       
 6   vaihteidenLkm         float64       
 7   ovienLukumaara        float64       
 8   istumapaikkojenLkm    float64       
 9   matkamittarilukema    float64       
dtypes: datetime64[ns](2), float64(8)
memory usage: 390.8 MB


,iskutilavuus,suurinNettoteho,sylintereidenLkm,omamassa,vaihteidenLkm,ovienLukumaara,istumapaikkojenLkm,matkamittarilukema
count,3.761320e+06,3.431689e+06,3.425262e+06,5.111370e+06,2.105992e+06,2.456816e+06,3.936116e+06,2.827326e+06
mean,2.124861e+03,9.822774e+01,3.873830e+00,1.664812e+03,5.832753e+00,4.211170e+00,4.141060e+00,2.034901e+05
std,6.532973e+03,5.443175e+01,4.325418e+00,2.579645e+03,3.122946e+00,8.452170e-01,2.647737e+00,5.782477e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,1.390000e+03,7.200000e+01,4.000000e+00,4.400000e+02,5.000000e+00,4.000000e+00,3.000000e+00,1.009550e+05
50%,1.798000e+03,9.190000e+01,4.000000e+00,1.375000e+03,6.000000e+00,4.000000e+00,5.000000e+00,1.805235e+05
75%,2.198000e+03,1.150000e+02,4.000000e+00,1.800000e+03,7.000000e+00,5.000000e+00,5.000000e+00,2.739440e+05
max,6.700000e+06,1.663500e+04,6.138000e+03,9.539560e+05,3.636000e+03,5.600000e+01,1.030000e+02,5.407337e+08


In [ ]:
text_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

In [ ]:
df["brand"] = df["merkkiSelvakielinen"]
df["model"] = df["mallimerkinta"]

In [ ]:
print(df["brand"].value_counts(dropna=False).head(20))
print(df["model"].value_counts(dropna=False).head(20))

brand
toyota            442880
volvo             295934
ford              264593
mercedes-benz     263777
volkswagen, vw    189455
muuli             180017
skoda             174880
volkswagen        172158
bmw               147263
nissan            143570
omavalmiste       129513
audi              125486
kia               118344
opel              110207
honda              92453
peugeot            87760
aku                86478
respo              78208
valmet             71355
majava             68350
Name: count, dtype: Int64
model
<NA>                                                  912979
toyota corolla farmari (ac) 4ov 1798cm3                17592
xc60 farmari (ac) 5ov 1969cm3 a                        16895
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3     16781
1250 xi/250                                            16596
transporter umpikorinen (bb) 6ov 1968cm3               16559
toyota rav4 farmari (ac) 4ov 2487cm3                   15680
toyota auris monikäyttöajoneuvo (af

In [ ]:
for col in ["ajoneuvoluokka", "ajoneuvoryhma"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- ajoneuvoluokka ---
ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
M3        8596
N2G       7458
T2        6591
L7e       4493
L6e       3546
N3G       2922
L2e       2282
M2        2263
L2        1320
L4         726
Name: count, dtype: int64

--- ajoneuvoryhma ---
ajoneuvoryhma
NaN            2778158
1, 13           603755
74              183494
75, 509         156317
2, 13           145061
109             138175
75               97355
509              94253
13, 20, 513      78761
6, 13            65029
107              62398
75, 933          49134
9, 13            47708
21               44689
7, 13            41542
73               33980
14               26529
13               24920
13, 20           24540
85         

## Filter to M1 passenger cars
Limit the cleaned registry table to M1 passenger cars before building summary features.

In [ ]:
car_df = df.copy()

car_df = car_df[
    car_df["ajoneuvoluokka"].notna()
].copy()

print(car_df["ajoneuvoluokka"].value_counts(dropna=False).head(20))

ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
Name: count, dtype: int64


In [ ]:
print(car_df["ajoneuvoluokka"].dropna().astype(str).str.strip().value_counts().head(20))

ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
Name: count, dtype: int64


In [ ]:
car_df["ajoneuvoluokka"] = car_df["ajoneuvoluokka"].astype("string").str.strip().str.lower()
m1_df = car_df[car_df["ajoneuvoluokka"] == "m1"].copy()
print(m1_df.shape)

(2565062, 27)


In [ ]:
for col in ["mallimerkinta", "kaupallinenNimi", "variantti", "versio"]:
    print(f"\n--- {col} ---")
    print(m1_df[col].value_counts(dropna=False).head(20))


--- mallimerkinta ---
mallimerkinta
toyota corolla farmari (ac) 4ov 1798cm3                    17552
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3         16779
toyota rav4 farmari (ac) 4ov 2487cm3                       15637
toyota auris monikäyttöajoneuvo (af) 4ov 1798cm3           15455
toyota yaris viistoperä (ab) 4ov 1490cm3                   14348
model y monikäyttöajoneuvo (af) 5ov                        14207
model 3 sedan (aa) 4ov                                     13802
toyota yaris monikäyttöajoneuvo (af) 4ov 1329cm3           13745
toyota c-hr viistoperä (ab) 4ov 1798cm3                    13324
v60 farmari (ac) 5ov 1969cm3 a                             11517
nissan qashqai monikäyttöajoneuvo (af) 4ov 1598cm3         11409
toyota avensis monikäyttöajoneuvo (af) 4ov 1798cm3         11269
fiesta viistoperä (ab) 4ov 998cm3                          10404
passat farmari (ac) 5ov 1395cm3 a                          10250
toyota yaris hybrid monikäyttöajoneuvo (af) 4ov 1497c

In [ ]:
m1_df.loc[~m1_df["iskutilavuus"].between(500, 10000, inclusive="both"), "iskutilavuus"] = pd.NA
m1_df.loc[~m1_df["suurinNettoteho"].between(20, 1000, inclusive="both"), "suurinNettoteho"] = pd.NA
m1_df.loc[~m1_df["sylintereidenLkm"].between(2, 16, inclusive="both"), "sylintereidenLkm"] = pd.NA
m1_df.loc[~m1_df["omamassa"].between(500, 5000, inclusive="both"), "omamassa"] = pd.NA
m1_df.loc[~m1_df["matkamittarilukema"].between(0, 1000000, inclusive="both"), "matkamittarilukema"] = pd.NA
m1_df.loc[~m1_df["ovienLukumaara"].between(2, 6, inclusive="both"), "ovienLukumaara"] = pd.NA
m1_df.loc[~m1_df["istumapaikkojenLkm"].between(1, 9, inclusive="both"), "istumapaikkojenLkm"] = pd.NA

In [ ]:
m1_df["model_family"] = (
    m1_df["kaupallinenNimi"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(m1_df["model_family"].value_counts(dropna=False).head(30))

model_family
golf              86763
octavia           82555
corolla           59454
focus             57105
<NA>              52363
passat            50893
nissan qashqai    47914
v70               43959
avensis           40308
toyota corolla    39081
toyota yaris      35843
ceed              35509
fiesta            31895
polo              31582
toyota auris      29253
toyota avensis    26980
toyota rav4       26596
mondeo            25838
v60               23427
superb            20840
yaris             20157
rio               20086
fabia             18620
s60               17703
toyota c-hr       16627
clio              15937
a4                15544
i20               14531
almera            14227
model y           14219
Name: count, dtype: Int64


In [ ]:
total_m1 = len(m1_df)
print(total_m1)

2565062


In [ ]:
print(m1_df.columns.tolist())

['merkkiSelvakielinen', 'mallimerkinta', 'kaupallinenNimi', 'variantti', 'versio', 'ensirekisterointipvm', 'kayttoonottopvm', 'ajoneuvoluokka', 'ajoneuvoryhma', 'korityyppi', 'kayttovoima', 'yksittaisKayttovoima', 'sahkohybridi', 'sahkohybridinluokka', 'iskutilavuus', 'suurinNettoteho', 'sylintereidenLkm', 'omamassa', 'vaihteisto', 'vaihteidenLkm', 'ahdin', 'ovienLukumaara', 'istumapaikkojenLkm', 'voimanvalJaTehostamistapa', 'matkamittarilukema', 'brand', 'model', 'model_family']


In [ ]:
registration_year = m1_df["ensirekisterointipvm"].dt.year
use_year = m1_df["kayttoonottopvm"].dt.year
vehicle_year = use_year.fillna(registration_year)

m1_df["registration_year"] = registration_year
m1_df["use_year"] = use_year
m1_df["vehicle_year"] = vehicle_year

print("Raw date sample:")
print(raw_date_sample)
print("\nParsed date sample:")
print(m1_df[["ensirekisterointipvm", "kayttoonottopvm"]].head(10))
print("\nregistration_year.describe()")
print(m1_df["registration_year"].describe())
print("\nuse_year.describe()")
print(m1_df["use_year"].describe())
print("\nvehicle_year.describe() before age calculation")
print(m1_df["vehicle_year"].describe())
print("\nregistration_year value_counts()")
print(m1_df["registration_year"].value_counts(dropna=False).head(20))
print("\nvehicle_year value_counts()")
print(m1_df["vehicle_year"].value_counts(dropna=False).head(20))

Raw date sample:
  ensirekisterointipvm  kayttoonottopvm
0                  NaN         19670000
1           01.09.1976         19760000
2           22.09.1983         19830000
3           09.02.1994         19940209
4           02.10.2003         20031002
5           17.03.2006         20060317
6           05.01.2007         20070105
7           05.05.1995         19950505
8           14.03.1996         19960314
9           21.05.1979         19790000

Parsed date sample:
   ensirekisterointipvm kayttoonottopvm
4            2003-10-02      2003-10-02
5            2006-03-17      2006-03-17
6            2007-01-05      2007-01-05
8            1996-03-14      1996-03-14
10           2000-03-24      2000-03-24
12                  NaT             NaT
19           1999-06-21      1999-06-21
22           1993-11-16      1993-11-16
31           2006-01-10      2006-01-10
32           2008-01-22      2008-01-22

registration_year.describe()
count    2.555171e+06
mean     2.013579e+03
std     

In [ ]:
reference_year = 2025
m1_df.loc[~m1_df["vehicle_year"].between(1950, 2025, inclusive="both"), "vehicle_year"] = pd.NA
m1_df["vehicle_age"] = reference_year - m1_df["vehicle_year"]
m1_df.loc[~m1_df["vehicle_age"].between(0, 80, inclusive="both"), "vehicle_age"] = pd.NA

print("\nvehicle_year.describe() after validity filter")
print(m1_df["vehicle_year"].describe())
print("\nvehicle_age.describe()")
print(m1_df["vehicle_age"].describe())


vehicle_year.describe() after validity filter
count    2.557839e+06
mean     2.012652e+03
std      7.983035e+00
min      1.950000e+03
25%      2.007000e+03
50%      2.014000e+03
75%      2.019000e+03
max      2.025000e+03
Name: vehicle_year, dtype: float64

vehicle_age.describe()
count    2.557839e+06
mean     1.234808e+01
std      7.983035e+00
min      0.000000e+00
25%      6.000000e+00
50%      1.100000e+01
75%      1.800000e+01
max      7.500000e+01
Name: vehicle_age, dtype: float64


In [ ]:
def make_year_band(y):
    if pd.isna(y):
        return pd.NA
    y = int(y)
    if y < 2005:
        return "pre_2005"
    elif y <= 2009:
        return "2005_2009"
    elif y <= 2014:
        return "2010_2014"
    elif y <= 2019:
        return "2015_2019"
    else:
        return "2020_2025"

m1_df["year_band"] = m1_df["vehicle_year"].apply(make_year_band)

## Standardize grouping keys
Create cleaned brand and model-family grouping keys before rebuilding the final summary tables.

In [ ]:
# Standardize brand labels for grouping.
brand_clean = (
    m1_df["brand"]
    .astype("string")
    .str.strip()
    .str.lower()
)

brand_clean = brand_clean.replace({
    "vw": "volkswagen",
    "volkswagen, vw": "volkswagen",
    "skd": "skoda",
    "skida": "skoda",
})

m1_df["brand_clean"] = brand_clean

In [ ]:
# Standardize model-family labels for grouping.
model_family_clean = (
    m1_df["model_family"]
    .astype("string")
    .str.strip()
    .str.lower()
)

for prefix in ["toyota ", "volkswagen ", "vw ", "skoda "]:
    model_family_clean = model_family_clean.str.removeprefix(prefix)

m1_df["model_family_clean"] = model_family_clean

In [ ]:
# Check that obvious duplicate brand labels collapse into one cleaned key.
brand_key_check = (
    m1_df.loc[
        m1_df["brand"].astype("string").str.contains("vw|volkswagen|skd|skida|skoda", case=False, na=False),
        ["brand", "brand_clean"],
    ]
    .drop_duplicates()
    .sort_values(["brand_clean", "brand"])
)

brand_key_check.head(20)

In [ ]:
# Check that brand-prefixed model labels collapse into one cleaned model family.
model_key_check = (
    m1_df.loc[
        m1_df["model_family"].astype("string").str.contains("corolla|octavia|golf", case=False, na=False),
        ["brand_clean", "model_family", "model_family_clean"],
    ]
    .drop_duplicates()
    .sort_values(["brand_clean", "model_family_clean", "model_family"])
)

model_key_check.head(30)

In [ ]:
# Legacy draft brand summary kept for reference.
# The final brand_summary is rebuilt below using cleaned grouping keys.

In [ ]:
# Export is intentionally deferred.
# Keep brand_summary in memory only for now.

## Build brand-level summary
Rebuild the final brand-level summary using the cleaned brand grouping key `brand_clean`.

In [ ]:
# Recreate cleaned grouping keys here if this section is run out of order.
if "brand_clean" not in m1_df.columns:
    brand_clean = (
        m1_df["brand"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "vw": "volkswagen",
            "volkswagen, vw": "volkswagen",
            "skd": "skoda",
            "skida": "skoda",
        })
    )
    m1_df["brand_clean"] = brand_clean

# Check which final brand_summary inputs already exist in the cleaned M1 table.
brand_summary_needed_cols = [
    "brand_clean",
    "vehicle_age",
    "matkamittarilukema",
    "iskutilavuus",
    "suurinNettoteho",
    "omamassa",
    "vaihteisto",
    "kayttovoima",
    "sahkohybridi",
    "ahdin",
]

brand_summary_available_cols = [col for col in brand_summary_needed_cols if col in m1_df.columns]
brand_summary_missing_cols = [col for col in brand_summary_needed_cols if col not in m1_df.columns]

print("Available brand_summary columns:", brand_summary_available_cols)
if brand_summary_missing_cols:
    print("Skipping missing brand_summary columns:", brand_summary_missing_cols)

In [ ]:
# Total M1 count is the denominator for market-share features.
total_m1 = len(m1_df)
print(total_m1)


2565062


In [ ]:
# Build the final brand-level summary using the cleaned brand key.
brand_summary_aggs = {
    "brand_total_registered": ("brand_clean", "size"),
}

if "vehicle_age" in m1_df.columns:
    brand_summary_aggs["brand_median_vehicle_age"] = ("vehicle_age", "median")
    brand_summary_aggs["brand_mean_vehicle_age"] = ("vehicle_age", "mean")

if "matkamittarilukema" in m1_df.columns:
    brand_summary_aggs["brand_median_mileage"] = ("matkamittarilukema", "median")
    brand_summary_aggs["brand_mean_mileage"] = ("matkamittarilukema", "mean")

if "iskutilavuus" in m1_df.columns:
    brand_summary_aggs["brand_median_engine_cc"] = ("iskutilavuus", "median")

if "suurinNettoteho" in m1_df.columns:
    brand_summary_aggs["brand_median_power_kw"] = ("suurinNettoteho", "median")

if "omamassa" in m1_df.columns:
    brand_summary_aggs["brand_median_mass_kg"] = ("omamassa", "median")

brand_summary = (
    m1_df.groupby("brand_clean", dropna=False)
    .agg(**brand_summary_aggs)
    .reset_index()
    .rename(columns={"brand_clean": "brand"})
)

In [ ]:
# Add brand-level share and composition features using cleaned brand labels.
brand_summary["brand_share_of_market"] = brand_summary["brand_total_registered"] / total_m1

if "vehicle_age" in m1_df.columns:
    over_10y = (m1_df["vehicle_age"] > 10).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_share_over_10y"] = brand_summary["brand"].map(over_10y)

if "matkamittarilukema" in m1_df.columns:
    over_200k_km = (m1_df["matkamittarilukema"] > 200000).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_share_over_200k_km"] = brand_summary["brand"].map(over_200k_km)

if "vaihteisto" in m1_df.columns:
    automatic_mask = m1_df["vaihteisto"].astype("string").isin(["2", "Y"]).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_automatic_share"] = brand_summary["brand"].map(automatic_mask)

if "kayttovoima" in m1_df.columns:
    powertrain = m1_df["kayttovoima"].astype("string")
    petrol_share = powertrain.eq("01").groupby(m1_df["brand_clean"], dropna=False).mean()
    diesel_share = powertrain.eq("02").groupby(m1_df["brand_clean"], dropna=False).mean()
    ev_share = powertrain.eq("04").groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_petrol_share"] = brand_summary["brand"].map(petrol_share)
    brand_summary["brand_diesel_share"] = brand_summary["brand"].map(diesel_share)
    brand_summary["brand_ev_share"] = brand_summary["brand"].map(ev_share)

if "sahkohybridi" in m1_df.columns:
    hybrid_mask = m1_df["sahkohybridi"].astype("string").eq("True").groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_hybrid_share"] = brand_summary["brand"].map(hybrid_mask)

if "ahdin" in m1_df.columns:
    turbo_mask = m1_df["ahdin"].astype("string").eq("True").groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_turbo_share"] = brand_summary["brand"].map(turbo_mask)

brand_summary = brand_summary.sort_values("brand_total_registered", ascending=False).reset_index(drop=True)

In [ ]:
# Quick preview of the final cleaned brand_summary.
brand_summary.head()

In [ ]:
# Legacy draft model summary kept for reference.
# The final model_summary is rebuilt below using cleaned grouping keys.

In [ ]:
# Legacy draft model summary kept for reference.
# The final model_summary is rebuilt below using cleaned grouping keys.

In [ ]:
# Legacy draft model summary kept for reference.
# The final model_summary is rebuilt below using cleaned grouping keys.

## Build model-level summary
Rebuild the final model-level summary using the cleaned grouping keys `brand_clean` and `model_family_clean`.

In [ ]:
# Recreate cleaned grouping keys here if this section is run out of order.
if "brand_clean" not in m1_df.columns:
    brand_clean = (
        m1_df["brand"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "vw": "volkswagen",
            "volkswagen, vw": "volkswagen",
            "skd": "skoda",
            "skida": "skoda",
        })
    )
    m1_df["brand_clean"] = brand_clean

if "model_family_clean" not in m1_df.columns:
    model_family_clean = (
        m1_df["model_family"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    for prefix in ["toyota ", "volkswagen ", "vw ", "skoda "]:
        model_family_clean = model_family_clean.str.removeprefix(prefix)

    m1_df["model_family_clean"] = model_family_clean

# Check which final model_summary inputs already exist in the cleaned M1 table.
model_summary_needed_cols = [
    "brand_clean",
    "model_family_clean",
    "vehicle_age",
    "matkamittarilukema",
    "iskutilavuus",
    "suurinNettoteho",
    "omamassa",
    "vaihteisto",
    "kayttovoima",
    "sahkohybridi",
    "ahdin",
]

model_summary_available_cols = [col for col in model_summary_needed_cols if col in m1_df.columns]
model_summary_missing_cols = [col for col in model_summary_needed_cols if col not in m1_df.columns]

print("Available model_summary columns:", model_summary_available_cols)
if model_summary_missing_cols:
    print("Skipping missing model_summary columns:", model_summary_missing_cols)

In [ ]:
# Total M1 count is the denominator for market-share features.
total_m1 = len(m1_df)
print(total_m1)

2565062


In [ ]:
# Compute cleaned brand totals for model_share_within_brand.
brand_totals = (
    m1_df.groupby("brand_clean", dropna=False)
    .size()
    .rename("brand_total_registered")
)

brand_totals.head()

In [ ]:
# Build the final model-level summary using cleaned grouping keys.
model_summary_aggs = {
    "model_total_registered": ("brand_clean", "size"),
}

if "vehicle_age" in m1_df.columns:
    model_summary_aggs["model_median_vehicle_age"] = ("vehicle_age", "median")
    model_summary_aggs["model_mean_vehicle_age"] = ("vehicle_age", "mean")

if "matkamittarilukema" in m1_df.columns:
    model_summary_aggs["model_median_mileage"] = ("matkamittarilukema", "median")
    model_summary_aggs["model_mean_mileage"] = ("matkamittarilukema", "mean")

if "iskutilavuus" in m1_df.columns:
    model_summary_aggs["model_median_engine_cc"] = ("iskutilavuus", "median")

if "suurinNettoteho" in m1_df.columns:
    model_summary_aggs["model_median_power_kw"] = ("suurinNettoteho", "median")

if "omamassa" in m1_df.columns:
    model_summary_aggs["model_median_mass_kg"] = ("omamassa", "median")

model_summary = (
    m1_df.groupby(["brand_clean", "model_family_clean"], dropna=False)
    .agg(**model_summary_aggs)
    .reset_index()
    .rename(columns={"brand_clean": "brand", "model_family_clean": "model_family"})
)

In [ ]:
# Add model-level share and composition features using cleaned grouping keys.
model_summary["model_share_of_market"] = model_summary["model_total_registered"] / total_m1
model_summary["model_share_within_brand"] = (
    model_summary["model_total_registered"]
    / model_summary["brand"].map(brand_totals)
)

model_group = [m1_df["brand_clean"], m1_df["model_family_clean"]]

if "vehicle_age" in m1_df.columns:
    over_10y = (m1_df["vehicle_age"] > 10).groupby(model_group, dropna=False).mean()
    model_summary["model_share_over_10y"] = model_summary.set_index(["brand", "model_family"]).index.map(over_10y)

if "matkamittarilukema" in m1_df.columns:
    over_200k_km = (m1_df["matkamittarilukema"] > 200000).groupby(model_group, dropna=False).mean()
    model_summary["model_share_over_200k_km"] = model_summary.set_index(["brand", "model_family"]).index.map(over_200k_km)

if "vaihteisto" in m1_df.columns:
    automatic_mask = m1_df["vaihteisto"].astype("string").isin(["2", "Y"]).groupby(model_group, dropna=False).mean()
    model_summary["model_automatic_share"] = model_summary.set_index(["brand", "model_family"]).index.map(automatic_mask)

if "kayttovoima" in m1_df.columns:
    powertrain = m1_df["kayttovoima"].astype("string")
    petrol_share = powertrain.eq("01").groupby(model_group, dropna=False).mean()
    diesel_share = powertrain.eq("02").groupby(model_group, dropna=False).mean()
    ev_share = powertrain.eq("04").groupby(model_group, dropna=False).mean()
    model_summary["model_petrol_share"] = model_summary.set_index(["brand", "model_family"]).index.map(petrol_share)
    model_summary["model_diesel_share"] = model_summary.set_index(["brand", "model_family"]).index.map(diesel_share)
    model_summary["model_ev_share"] = model_summary.set_index(["brand", "model_family"]).index.map(ev_share)

if "sahkohybridi" in m1_df.columns:
    hybrid_mask = m1_df["sahkohybridi"].astype("string").eq("True").groupby(model_group, dropna=False).mean()
    model_summary["model_hybrid_share"] = model_summary.set_index(["brand", "model_family"]).index.map(hybrid_mask)

if "ahdin" in m1_df.columns:
    turbo_mask = m1_df["ahdin"].astype("string").eq("True").groupby(model_group, dropna=False).mean()
    model_summary["model_turbo_share"] = model_summary.set_index(["brand", "model_family"]).index.map(turbo_mask)

model_summary = model_summary.sort_values(
    ["model_total_registered", "brand", "model_family"],
    ascending=[False, True, True],
).reset_index(drop=True)

## Inspect target models
Check that the final cleaned summary tables collapse the target brands and models into consistent labels.

In [ ]:
# Final brand_summary preview.
brand_summary.head()

In [ ]:
# Final brand_summary shape.
brand_summary.shape

In [ ]:
# Final model_summary preview.
model_summary.head()

In [ ]:
# Final model_summary shape.
model_summary.shape

In [ ]:
# Preview cleaned target brands and models if they are present.
target_models = model_summary[
    ((model_summary["brand"] == "toyota") & (model_summary["model_family"] == "corolla"))
    | ((model_summary["brand"] == "skoda") & (model_summary["model_family"] == "octavia"))
    | ((model_summary["brand"] == "volkswagen") & (model_summary["model_family"] == "golf"))
]

target_models.head(20)